# When does a translation change what someone can do?

A person writing to their manager may care more about keeping a shift than about which English words the translator chose. This notebook first calculates how a trained model’s translation scores changed across domains. It then works through a newly written Polish message in which one time constraint determines whether a manager can offer the shift.

## Replay recorded scores and try a shift decision on CPU

**Run all cells** using any Python 3.9+ kernel, including a standard Colab CPU runtime. The cells use the Python standard library and embedded inputs. Training, model inference and COMET scoring took place in the original experiments.

## What we trained and measured

We fine-tuned **NLLB-200-distilled-600M** and **Gemma 3 1B-IT** with LoRA. Each full run used 100,000 Polish, 100,000 Croatian and 100,000 Russian source–English pairs. Polish came from JRC-Acquis and Europarl, Croatian from DGT Translation Memory, and Russian from the United Nations Parallel Corpus. The data preparation grouped official text by document to protect the held-out splits. The Gemma data revision replaced 576 malformed training records while retaining the same language quotas. We kept Ukrainian out of adapter training to test transfer.

NLLB’s base and adapted translations were compared on held-out documents, everyday messages and Ukrainian transfer data. The Gemma comparison evaluated its base and three checkpoints, saved after 10,000, 50,000 and 300,000 official pairs. It used BLEU, chrF++ and reference-based COMET where an English reference existed. Sixteen previously examined stress cases had questions about the reader’s decision and no reference translation. The original runs are documented in this repository’s README.

This independent project was developed during the [Google DeepMind AI Research Foundations capstone](https://www.skills.google/paths/3135/course_templates/1557).

## Compare Gemma’s COMET scores by language and domain

The eight rows below contain paired **Gemma** results with the same messages at baseline and after 300,000 training examples. The official test used 32 examples for each trained language. The Ukrainian transfer test used 32 FLORES-200 examples. The everyday set used 18 messages per language. Each COMET entry is the mean across the cases in that row. Higher scores indicate closer agreement with the source and the English reference as judged by the metric.

The code subtracts the base mean from the adapted mean. All inputs are embedded so this cell also works when Colab opens the notebook on its own. The repository provides the [complete aggregate CSV](https://github.com/Silveroboros-dev/slavic-english-slm/blob/main/results/metrics.csv), including the intermediate checkpoints.

In [1]:
# WMT22 COMET-DA means from results/metrics.csv, with full recorded precision.
# Each row holds the panel, language, cases, base mean and final mean.
observations = [
    ("official_test", "pl", 32, 0.774580979719758, 0.8383090831339359),
    ("official_test", "hr", 32, 0.7792420759797096, 0.8653117902576923),
    ("official_test", "ru", 32, 0.8303902745246887, 0.8547991644591093),
    ("uk_transfer", "uk", 32, 0.8279979377985001, 0.8257076106965542),
    ("everyday_regression", "pl", 18, 0.6742872413661745, 0.6372793250613742),
    ("everyday_regression", "hr", 18, 0.7482771045631833, 0.7699387967586517),
    ("everyday_regression", "ru", 18, 0.7723085201448865, 0.7452084388997819),
    ("everyday_regression", "uk", 18, 0.7885835667451223, 0.7569239205784268),
]

print(
    f'{"Panel":<21} {"Language":<9} {"Cases":>5} '
    f'{"Base":>8} {"Final":>8} {"Change":>8}'
)
for panel, language, cases, base, final in observations:
    print(
        f"{panel:<21} {language:<9} {cases:>5} "
        f"{base:>8.4f} {final:>8.4f} {final - base:>+8.4f}"
    )

Panel                 Language  Cases     Base    Final   Change
official_test         pl           32   0.7746   0.8383  +0.0637
official_test         hr           32   0.7792   0.8653  +0.0861
official_test         ru           32   0.8304   0.8548  +0.0244
uk_transfer           uk           32   0.8280   0.8257  -0.0023
everyday_regression   pl           18   0.6743   0.6373  -0.0370
everyday_regression   hr           18   0.7483   0.7699  +0.0217
everyday_regression   ru           18   0.7723   0.7452  -0.0271
everyday_regression   uk           18   0.7886   0.7569  -0.0317


Official-test COMET increased for all three training languages. On everyday messages it rose for Croatian and fell for Polish, Russian and Ukrainian. The Croatian change was +0.0217. Its paired 95% resampling interval ran from −0.0188 to +0.0590 across 18 messages. The 32-item Ukrainian FLORES comparison was close to flat. These figures describe one training run and small, fixed sets of messages.

## Can the manager offer the 08:15 shift?

**Polish source:** „W sobotę mogę zacząć zmianę w piekarni najwcześniej o 7:45. Muszę skończyć do 11:00.”

A bakery worker can start at **07:45 or later** on Saturday and must finish by **11:00**. Could the manager offer **08:15–10:30**? The answer implied by the Polish message is **yes**.

We wrote both English alternatives below for this demonstration. They are illustrative examples independent of the recorded model outputs and evaluation sets. Each line pairs an English message with an explicit interpretation of its start time. The calculation asks which offers fit each interpretation. A human reader would supply the interpretation in a comprehension study.

In [2]:
from datetime import time

# The second value is a declared reading of the English message.
earliest = (
    "On Saturday I can start at the bakery at 07:45 at the earliest. "
    "I have to finish by 11:00.",
    "earliest",
)
fixed = (
    "On Saturday I can start at the bakery only at 07:45. "
    "I have to finish by 11:00.",
    "fixed",
)
alternatives = {"A": earliest, "B": fixed}

for label, (english, start_rule) in alternatives.items():
    print(f"Version {label}: {english}")

Version A: On Saturday I can start at the bakery at 07:45 at the earliest. I have to finish by 11:00.
Version B: On Saturday I can start at the bakery only at 07:45. I have to finish by 11:00.


In [3]:
def fits_shift(start_rule, offer_start, offer_end):
    """Apply the declared reading to a proposed shift on this Saturday."""
    start = time.fromisoformat(offer_start)
    end = time.fromisoformat(offer_end)
    boundary = time(7, 45)
    if start_rule == "earliest":
        valid_start = start >= boundary
    elif start_rule == "fixed":
        valid_start = start == boundary
    else:
        raise ValueError(f"Unknown start rule: {start_rule}")
    return valid_start and start < end <= time(11, 0)


# The rightmost value is the answer implied by the Polish source.
offers = [
    ("07:45", "11:00", True),
    ("08:15", "10:30", True),
    ("07:30", "10:30", False),
    ("08:15", "11:15", False),
]
yes_no = {True: "yes", False: "no"}
changed_offers = {label: [] for label in alternatives}
print("Offer        Source   Version A   Version B")
for start, end, expected in offers:
    decisions = {}
    for label, (_, rule) in alternatives.items():
        decisions[label] = fits_shift(rule, start, end)
        if decisions[label] != expected:
            changed_offers[label].append((start, end))
    print(f"{start}–{end}  {yes_no[expected]:<8} "
          f"{yes_no[decisions['A']]:<11} {yes_no[decisions['B']]}")

assert changed_offers == {"A": [], "B": [("08:15", "10:30")]}

Offer        Source   Version A   Version B
07:45–11:00  yes      yes         yes
08:15–10:30  yes      yes         no
07:30–10:30  no       no          no
08:15–11:15  no       no          no


Both readings allow the shift that begins exactly at 07:45. That offer alone would hide the error. The later start at 08:15 exposes it: the worker is available, but a manager relying on version B could reject the offer. The code checks the consequences of the two supplied interpretations. Estimating the frequency of such errors would require independent readers and a representative sample.

To test a real translation, a reader could see one anonymized English version and a proposed shift, then answer whether that shift is possible. Compare their answer with the intended action recorded from the source speaker. Fresh scenarios should stay together across variants and remain outside training and model selection.